# 03 - Matryoshka Representation Fine-Tuning
**Domain Adaptation via Multi-Granularity Contrastive Learning**

In this notebook:
1. Wrap `MultipleNegativesRankingLoss` (MNRL) inside `MatryoshkaLoss`.
2. Jointly optimize sub-dimension slices: `[768, 512, 256, 128]`.
3. Train for 2 epochs on Colab T4 GPU with mixed precision (FP16).
4. Save fine-tuned checkpoints directly to Google Drive.


In [18]:
import os

# 1. Clone only if not already cloned
if not os.path.exists("matryoshka-domain-rag") and not os.path.exists("src"):
    !git clone https://github.com/premsaipusapati-debug/matryoshka-domain-rag.git
    %cd matryoshka-domain-rag
elif os.path.exists("matryoshka-domain-rag"):
    %cd matryoshka-domain-rag

# 2. Pull any recent changes from GitHub
!git pull

# 3. Install requirements
!pip install -q -r requirements.txt


Already up to date.


In [19]:
import sys
import os
import torch

if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.append(os.path.abspath(".."))
else:
    sys.path.append(os.getcwd())

import yaml
from src.data_loader import load_scifact_raw, create_training_data, create_training_dataloader
from src.model import load_embedding_model
from src.trainer import train_matryoshka_model

assert torch.cuda.is_available(), "CUDA GPU is required for training! Please select Runtime > Change Runtime type > T4 GPU."
print(f"Training on GPu: {torch.cuda.get_device_name(0)}")

# Load configuration
config_path = "../configs/config.yaml" if os.path.exists("../configs/config.yaml") else "configs/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

Training on GPu: Tesla T4


In [20]:
raw_data = load_scifact_raw(config["dataset"]["name"])
train_examples = create_training_data(raw_data, max_samples=config["dataset"]["max_train_samples"])

train_loader = create_training_dataloader(
    train_examples,
    batch_size=config["training"]['train_batch_size']
)

print(f"Prepared {len(train_examples)} training pairs")
print(f"DataLoader; {len(train_loader)} batched per epoch")

Prepared 919 training pairs
DataLoader; 28 batched per epoch


In [21]:
base_model = load_embedding_model(
    model_name_or_path=config["model"]["base_model"],
    max_seq_length=config["model"]["max_seq_length"]
)

#Output directory on Google Drive or local output
checkpoint_dir = "/content/drive/MyDrive/Matryoshka-RAG/bge-scifact-metryoshka"
if not os.path.exists("/content/drive/MyDrive"):
  checkpoint_dir = "../output/bge-scifact-matryoshka"

os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {checkpoint_dir}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Checkpoints will be saved to: ../output/bge-scifact-matryoshka


In [22]:
import time

start_time= time.time()
fine_tuned_model = train_matryoshka_model(
    model=base_model,
    train_dataloader=train_loader,
    config=config,
    output_path=checkpoint_dir
)

elpased = (time.time() - start_time) / 60
print(f"\n✅ Training completed in {elapsed:.2f} minutes!")
print(f"Model saved to: {checkpoint_dir}")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'elapsed' is not defined